## 02a_claims_transform — RCM Intelligence Platform
### Purpose
Transforms raw Bronze inpatient claims into a clean, enriched
Silver table with RCM-specific features engineered and ready
for dimensional modeling in Gold.

### What this does
1. Reads raw inpatient claims from Bronze
2. Casts all columns to correct data types
3. Standardises and cleans provider/DRG fields
4. Engineers core RCM features:
   - Charge-to-payment ratio (CTP)
   - Medicare payment percentage
   - Patient responsibility amount
   - Underpayment flag
   - Revenue gap
   - High volume flag
   - DRG severity tier
5. Joins hospital dimension for enrichment
6. Runs DQ checks — failed rows go to quarantine
7. Writes to Silver Delta table
8. Logs run to audit table

### Inputs
- rcm_platform.rcm_bronze.inpatient_claims_raw
- rcm_platform.rcm_bronze.hospital_info_raw

### Outputs
- rcm_platform.rcm_silver.inpatient_claims_enriched

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Silver |
| Run order  | 4 — after 01b_ingest_static_files |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
from datetime import datetime, timezone

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "02a_claims_transform"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")

In [0]:
# ============================================================
# STEP 1 — READ BRONZE AND CAST TO CORRECT TYPES
# Bronze stores everything as string — Silver applies types
# ============================================================

print("=" * 55)
print("  READING BRONZE INPATIENT CLAIMS")
print("=" * 55)

df_bronze = spark.table(TBL_BRONZE_INPATIENT_RAW)
bronze_count = df_bronze.count()
print(f"Bronze rows     : {bronze_count:,}")
print(f"Bronze columns  : {len(df_bronze.columns)}")

# Preview raw column names from CMS
print("\nCMS column names:")
non_meta = [c for c in df_bronze.columns if not c.startswith("_")]
for c in non_meta:
    print(f"  {c}")

In [0]:
# ============================================================
# STEP 3 — BUILD SILVER CLAIMS DATAFRAME
# Using expr + try_cast to handle empty strings gracefully
# ============================================================

from pyspark.sql.functions import expr, trim, col
from pyspark.sql.types import IntegerType

df_claims = df_bronze.select(
    trim(col("Rndrng_Prvdr_CCN")).alias("provider_id"),
    trim(col("Rndrng_Prvdr_Org_Name")).alias("provider_name"),
    trim(col("Rndrng_Prvdr_St")).alias("provider_street"),
    trim(col("Rndrng_Prvdr_City")).alias("provider_city"),
    trim(col("Rndrng_Prvdr_State_Abrvtn")).alias("provider_state"),
    trim(col("Rndrng_Prvdr_State_FIPS")).alias("provider_state_fips"),
    trim(col("Rndrng_Prvdr_Zip5")).alias("provider_zip"),
    trim(col("Rndrng_Prvdr_RUCA")).alias("ruca_code"),
    trim(col("Rndrng_Prvdr_RUCA_Desc")).alias("ruca_description"),
    trim(col("DRG_Cd")).alias("drg_code"),
    trim(col("DRG_Desc")).alias("drg_description"),
    expr("try_cast(Tot_Dschrgs as int)").alias("total_discharges"),
    expr("try_cast(Avg_Submtd_Cvrd_Chrg as double)").alias("avg_submitted_charge"),
    expr("try_cast(Avg_Tot_Pymt_Amt as double)").alias("avg_total_payment"),
    expr("try_cast(Avg_Mdcr_Pymt_Amt as double)").alias("avg_medicare_payment"),
    col("_batch_id"),
    col("_batch_date"),
    col("_ingested_at")
)

raw_silver_count = df_claims.count()
print(f"Rows after select : {raw_silver_count:,}")
print(f"Columns           : {len(df_claims.columns)}")
display(df_claims.limit(5))

In [0]:
# ============================================================
# STEP 4 — ENGINEER RCM FEATURES
# These are the metrics that matter to hospital finance teams
# ============================================================

df_features = df_claims \
    .withColumn("charge_to_payment_ratio",
        when(col("avg_total_payment") > 0,
            round(col("avg_submitted_charge") / col("avg_total_payment"), 4)
        ).otherwise(lit(None))
    ) \
    .withColumn("medicare_payment_pct",
        when(col("avg_total_payment") > 0,
            round(col("avg_medicare_payment") / col("avg_total_payment") * 100, 2)
        ).otherwise(lit(None))
    ) \
    .withColumn("patient_responsibility",
        round(col("avg_total_payment") - col("avg_medicare_payment"), 2)
    ) \
    .withColumn("revenue_gap",
        round(col("avg_submitted_charge") - col("avg_total_payment"), 2)
    ) \
    .withColumn("revenue_gap_pct",
        when(col("avg_submitted_charge") > 0,
            round(
                (col("avg_submitted_charge") - col("avg_total_payment"))
                / col("avg_submitted_charge") * 100, 2
            )
        ).otherwise(lit(None))
    ) \
    .withColumn("underpayment_flag",
        when(
            (col("avg_medicare_payment") > 0) &
            (col("avg_total_payment") > 0) &
            (col("avg_medicare_payment") < col("avg_total_payment") * 0.80),
            lit(1)
        ).otherwise(lit(0))
    ) \
    .withColumn("high_volume_flag",
        when(col("total_discharges") > 100, lit(1)).otherwise(lit(0))
    ) \
    .withColumn("drg_severity_tier",
        when(expr("try_cast(drg_code as double)") < 100,  lit("Pre-MDC"))
        .when(expr("try_cast(drg_code as double)").between(100, 299), lit("Surgical"))
        .when(expr("try_cast(drg_code as double)").between(300, 499), lit("Medical"))
        .when(expr("try_cast(drg_code as double)").between(500, 699), lit("Specialty"))
        .when(expr("try_cast(drg_code as double)").between(700, 899), lit("Mental Health"))
        .otherwise(lit("Other"))
    ) \
    .withColumn("total_revenue_gap",
        round(col("revenue_gap") * col("total_discharges"), 2)
    ) \
    .withColumn("rural_urban_classification",
        when(expr("try_cast(ruca_code as double)") == 1.0,  lit("Urban Core"))
        .when(expr("try_cast(ruca_code as double)").between(1.1, 3.9), lit("Urban"))
        .when(expr("try_cast(ruca_code as double)").between(4.0, 6.9), lit("Large Rural"))
        .when(expr("try_cast(ruca_code as double)").between(7.0, 9.9), lit("Small Rural"))
        .when(expr("try_cast(ruca_code as double)") == 10.0, lit("Remote Rural"))
        .otherwise(lit("Unknown"))
    ) \
    .withColumn("_silver_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_silver_batch_id",     lit(BATCH_ID))

feature_count = df_features.count()
print(f"Rows with features : {feature_count:,}")
print(f"Columns            : {len(df_features.columns)}")
print(f"\nRCM feature preview:")
display(
    df_features.select(
        "provider_id",
        "drg_code",
        "avg_submitted_charge",
        "avg_total_payment",
        "charge_to_payment_ratio",
        "revenue_gap",
        "revenue_gap_pct",
        "underpayment_flag",
        "drg_severity_tier",
        "rural_urban_classification"
    ).limit(5)
)

In [0]:
# ============================================================
# STEP 5 — ENRICH WITH HOSPITAL DIMENSION
# Join hospital info to add type, ownership and location context
# ============================================================

print("=" * 55)
print("  JOINING HOSPITAL DIMENSION")
print("=" * 55)

df_hospitals = spark.table(TBL_BRONZE_HOSPITAL_RAW).select(
    col("facility_id").alias("provider_id"),
    col("hospital_type"),
    col("hospital_ownership"),
    col("emergency_services"),
    col("hospital_overall_rating"),
    col("county_parish").alias("county"),
    col("birthing_friendly").alias("birthing_friendly")
        if "birthing_friendly" in spark.table(TBL_BRONZE_HOSPITAL_RAW).columns
        else col("meets_criteria_for_birthing_friendly_designation")
            .alias("birthing_friendly")
)

df_enriched = df_features.join(
    df_hospitals,
    on  = "provider_id",
    how = "left"
)

joined_count  = df_enriched.count()
matched_count = df_enriched.filter(col("hospital_type").isNotNull()).count()
unmatched     = joined_count - matched_count

print(f"Rows after join     : {joined_count:,}")
print(f"Matched to hospital : {matched_count:,}")
print(f"Unmatched (no info) : {unmatched:,}")

display(
    df_enriched.select(
        "provider_id",
        "provider_name",
        "hospital_type",
        "hospital_ownership",
        "hospital_overall_rating",
        "emergency_services"
    ).limit(5)
)

In [0]:
# ============================================================
# STEP 6 — RUN DATA QUALITY CHECKS
# Failed rows go to quarantine — never silently dropped
# ============================================================

print("=" * 55)
print("  RUNNING DATA QUALITY CHECKS")
print("=" * 55)

df_clean = run_dq_checks(
    df            = df_enriched,
    source_table  = TBL_BRONZE_INPATIENT_RAW,
    notebook_name = NOTEBOOK_NAME
)

quarantine_count = joined_count - df_clean.count()
clean_count      = df_clean.count()

print(f"\nRows passing DQ : {clean_count:,}")
print(f"Rows quarantined: {quarantine_count:,}")

In [0]:
# ============================================================
# STEP 7 — WRITE TO SILVER DELTA TABLE (direct write)
# ============================================================

print("=" * 55)
print("  WRITING TO SILVER DELTA TABLE")
print("=" * 55)

# Write directly - columns are already correctly typed from earlier steps
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_SILVER_CLAIMS)

silver_rows = spark.table(TBL_SILVER_CLAIMS).count()

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {TBL_SILVER_CLAIMS}
    IS 'Enriched Medicare inpatient claims with RCM features'
""")

# Run OPTIMIZE + ZORDER separately after write
print("Running OPTIMIZE...")
spark.sql(f"OPTIMIZE {TBL_SILVER_CLAIMS} ZORDER BY (provider_state, drg_code)")

print(f"\nSilver table : {TBL_SILVER_CLAIMS}")
print(f"Rows written : {silver_rows:,}")

display(
    spark.table(TBL_SILVER_CLAIMS).select(
        "provider_id",
        "provider_name",
        "provider_state",
        "drg_code",
        "avg_submitted_charge",
        "avg_medicare_payment",
        "charge_to_payment_ratio",
        "revenue_gap",
        "underpayment_flag",
        "drg_severity_tier",
        "hospital_type"
    ).limit(10)
)

In [0]:
# ============================================================
# STEP 8 — VERIFICATION
# ============================================================

print("=" * 55)
print("  SILVER VERIFICATION")
print("=" * 55)

spark.sql(f"""
    SELECT
        COUNT(*)                           AS total_rows,
        COUNT(DISTINCT provider_id)        AS unique_providers,
        COUNT(DISTINCT drg_code)           AS unique_drgs,
        COUNT(DISTINCT provider_state)     AS states_covered,
        SUM(underpayment_flag)             AS underpayment_flags,
        ROUND(AVG(charge_to_payment_ratio), 2) AS avg_ctp_ratio,
        ROUND(AVG(revenue_gap_pct), 2)     AS avg_revenue_gap_pct,
        ROUND(SUM(total_revenue_gap), 0)   AS total_revenue_gap_dollars
    FROM {TBL_SILVER_CLAIMS}
""").show(truncate=False)

print("\nRevenue gap by DRG severity tier:")
spark.sql(f"""
    SELECT
        drg_severity_tier,
        COUNT(*)                               AS claim_rows,
        ROUND(AVG(revenue_gap_pct), 2)         AS avg_revenue_gap_pct,
        ROUND(AVG(charge_to_payment_ratio), 2) AS avg_ctp_ratio,
        SUM(underpayment_flag)                 AS underpayment_count
    FROM {TBL_SILVER_CLAIMS}
    GROUP BY drg_severity_tier
    ORDER BY avg_revenue_gap_pct DESC
""").show(truncate=False)

print("\nTop 5 states by total revenue gap:")
spark.sql(f"""
    SELECT
        provider_state,
        COUNT(DISTINCT provider_id)            AS hospitals,
        ROUND(SUM(total_revenue_gap), 0)       AS total_revenue_gap,
        ROUND(AVG(charge_to_payment_ratio), 2) AS avg_ctp_ratio
    FROM {TBL_SILVER_CLAIMS}
    GROUP BY provider_state
    ORDER BY total_revenue_gap DESC
    LIMIT 5
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 9 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "silver",
    status           = "SUCCESS",
    rows_read        = bronze_count,
    rows_written     = silver_rows,
    rows_quarantined = quarantine_count,
    message          = (
        f"Batch {BATCH_ID} — "
        f"bronze: {bronze_count:,} | "
        f"silver: {silver_rows:,} | "
        f"quarantined: {quarantine_count:,} | "
        f"11 RCM features engineered | "
        f"OPTIMIZE + ZORDER on provider_state, drg_code"
    )
)